# Task 2b: Relation Extraction

In [1]:
from collections import Counter
from datasets import Dataset
import numpy as np
from pathlib import Path
import pandas as pd
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, set_seed, TrainingArguments, Trainer

In [2]:
JA_PATH = Path('train/ja_train')
JA_DEV = Path('dev/ja_dev')

In [3]:
corpus_train = {}
for f in JA_PATH.iterdir():
    if f.suffix == '.txt':
        with f.open() as text:
            fields = text.readlines()[0].split(':')
            keyid = fields[0]
            offset = len(keyid)+1
            data = ':'.join(fields[1:])
            if f.stem in corpus_train:
                corpus_train[f.stem]['text'] = data
                corpus_train[f.stem]['offset'] = len(keyid) + 1
            else:
                corpus_train[f.stem] = {'text': data,
                                  'offset': len(keyid) + 1,
                                  'ann': [],
                                  'rel': []}
    elif f.suffix == '.ann':
        with f.open() as anno:
            if f.stem not in corpus_train:
                corpus_train[f.stem] = {'ann': [], 'rel': []}
            for line in anno:
                fields = line.strip().split("\t")
                if line.startswith("T"):
                    corpus_train[f.stem]['ann'].append(fields)
                elif line.startswith("R"):
                    corpus_train[f.stem]['rel'].append(fields[1].split(' '))

In [4]:
corpus_dev = {}
for f in JA_DEV.iterdir():
    if f.suffix == '.txt':
        with f.open() as text:
            fields = text.readlines()[0].split(':')
            keyid = fields[0]
            offset = len(keyid)+1
            data = ':'.join(fields[1:])
            if f.stem in corpus_dev:
                corpus_dev[f.stem]['text'] = data
                corpus_dev[f.stem]['offset'] = len(keyid) + 1
            else:
                corpus_dev[f.stem] = {'text': data,
                                  'offset': len(keyid) + 1,
                                  'ann': [],
                                  'rel': []}
    elif f.suffix == '.ann':
        with f.open() as anno:
            if f.stem not in corpus_dev:
                corpus_dev[f.stem] = {'ann': [], 'rel': []}
            for line in anno:
                fields = line.strip().split("\t")
                if line.startswith("T"):
                    corpus_dev[f.stem]['ann'].append(fields)
                elif line.startswith("R"):
                    corpus_dev[f.stem]['rel'].append(fields[1].split(' '))

In [5]:
rel_ex_cnt = Counter()
for key, obj in corpus_train.items():
    for label, source, dest in obj['rel']:
        _source = source.split(':')[1]
        _dest = dest.split(':')[1]
        sourcemeta = [meta for tid, meta, _ in obj['ann'] if tid == _source][0].split(' ')[0]
        destmeta = [meta for tid, meta, _ in obj['ann'] if tid == _dest][0].split(' ')[0]
        rel_ex_cnt[f"{sourcemeta}\t{label}\t{destmeta}"] += 1

print("train set")
for rel, cnt in rel_ex_cnt.most_common():
        print(f"#relation {rel}: {cnt}")


rel_ex_cnt = Counter()
for key, obj in corpus_dev.items():
    for label, source, dest in obj['rel']:
        _source = source.split(':')[1]
        _dest = dest.split(':')[1]
        sourcemeta = [meta for tid, meta, _ in obj['ann'] if tid == _source][0].split(' ')[0]
        destmeta = [meta for tid, meta, _ in obj['ann'] if tid == _dest][0].split(' ')[0]
        rel_ex_cnt[f"{sourcemeta}\t{label}\t{destmeta}"] += 1

print("dev set")
for rel, cnt in rel_ex_cnt.most_common():
        print(f"#relation {rel}: {cnt}")

train set
#relation DRUG	CAUSED	DISORDER: 390
#relation DRUG	TREATMENT_FOR	DISORDER: 100
#relation DISORDER	CAUSED	DISORDER: 98
#relation DRUG	CAUSED	FUNCTION: 20
#relation DISORDER	CAUSED	FUNCTION: 8
#relation DRUG	TREATMENT_FOR	FUNCTION: 3
dev set
#relation DRUG	CAUSED	DISORDER: 189
#relation DRUG	TREATMENT_FOR	DISORDER: 38
#relation DISORDER	CAUSED	DISORDER: 29
#relation DISORDER	CAUSED	FUNCTION: 5
#relation DRUG	CAUSED	FUNCTION: 4
#relation DRUG	TREATMENT_FOR	FUNCTION: 1


## Find all possible spans between the NE

excluding the entities

In [6]:
train_dataset = []
for key, obj in corpus_train.items():
    classcorpus = []
    for aid, ameta, _ in obj['ann']:
        for bid, bmeta, _ in obj['ann']:
            if aid == bid:
                continue
            alabel, astart, aend = ameta.split(' ')
            blabel, bstart, bend = bmeta.split(' ')
            if (alabel == 'DRUG' and blabel !='DRUG') or (alabel == 'DISORDER' and blabel != 'DRUG'):
                start = min(int(aend), int(bend))
                end = max(int(astart), int(bstart))
                start -= obj['offset']
                end -= obj['offset']
                classcorpus.append([aid, bid, obj['text'][start: end], "O"])
    
    for label, source, dest in obj['rel']:
        _source = source.split(':')[1]
        _dest = dest.split(':')[1]
        for c in classcorpus:
            if c[0] == _source and c[1] == _dest:
                c[3] = label
    for _, _, _text, _label in classcorpus:
        train_dataset.append([_text, _label])

In [7]:
dev_dataset = []
for key, obj in corpus_dev.items():
    classcorpus = []
    for aid, ameta, _ in obj['ann']:
        for bid, bmeta, _ in obj['ann']:
            if aid == bid:
                continue
            alabel, astart, aend = ameta.split(' ')
            blabel, bstart, bend = bmeta.split(' ')
            if (alabel == 'DRUG' and blabel !='DRUG') or (alabel == 'DISORDER' and blabel != 'DRUG'):
                start = min(int(aend), int(bend))
                end = max(int(astart), int(bstart))
                start -= obj['offset']
                end -= obj['offset']
                classcorpus.append([aid, bid, obj['text'][start: end], "O"])
    
    for label, source, dest in obj['rel']:
        _source = source.split(':')[1]
        _dest = dest.split(':')[1]
        for c in classcorpus:
            if c[0] == _source and c[1] == _dest:
                c[3] = label
    for _, _, _text, _label in classcorpus:
        dev_dataset.append([_text, _label])

In [8]:
len(train_dataset), len(dev_dataset)

(8497, 3225)

In [9]:
train_df = pd.DataFrame(train_dataset, columns=['text', 'label'])
dev_df = pd.DataFrame(dev_dataset, columns=['text', 'label'])

## Base Model

In [10]:
set_seed(42)
model_checkpoint = "daisaku-s/medtxt_ner_roberta"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3, ignore_mismatched_sizes=True)

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at daisaku-s/medtxt_ner_roberta and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at daisaku-s/medtxt_ner_roberta and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([79, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([79]) in the ch

In [11]:
label2id = {'O': 0, 'CAUSED': 1, 'TREATMENT_FOR': 2}
id2label = ['O', 'CAUSED', 'TREATMENT_FOR']

def preprocess_data(examples):
    encoding = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=512)
    encoding["labels"] = [label2id[l] for l in examples["label"]]
    return encoding

## Metric

In [12]:
def compute_metrics(p):
    preds = p.predictions[0] if isinstance(p.predictions,
            tuple) else p.predictions
    # multi label
    y_pred = np.argmax(preds, axis=1)
    return {'f1': f1_score(p.label_ids, y_pred, average='macro')}

## Training Params

In [13]:
from transformers import TrainingArguments, Trainer
batch_size = 16
args = TrainingArguments(
    f"ja_class",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=15,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    #gradient_accumulation_steps=4,
    #gradient_checkpointing=True,
    save_total_limit=3,
    push_to_hub=False,
)


## Train on 80% train set

In [14]:
train80_df, val_df = train_test_split(train_df, test_size=0.2, random_state=42, shuffle=True, stratify=train_df['label'])

In [15]:
train_ds = Dataset.from_pandas(train80_df, preserve_index=False)
train_ds = train_ds.map(preprocess_data, batched=True, remove_columns=train_ds.column_names)
train_ds.set_format("torch")

valid_ds = Dataset.from_pandas(val_df, preserve_index=False)
valid_ds = valid_ds.map(preprocess_data, batched=True, remove_columns=valid_ds.column_names)
valid_ds.set_format("torch")

Map:   0%|          | 0/6797 [00:00<?, ? examples/s]

Map:   0%|          | 0/1700 [00:00<?, ? examples/s]

In [16]:
trainer = Trainer(
    model,
    args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,F1
1,No log,0.246126,0.392282
2,0.265900,0.272348,0.368221
3,0.211900,0.264295,0.497159
4,0.180400,0.273555,0.505275
5,0.146300,0.291540,0.563475
6,0.121100,0.312129,0.578628
7,0.121100,0.364200,0.513223
8,0.099700,0.361680,0.624104
9,0.093300,0.369153,0.624600
10,0.083300,0.390356,0.604276


TrainOutput(global_step=6375, training_loss=0.1198987268186083, metrics={'train_runtime': 2974.8927, 'train_samples_per_second': 34.272, 'train_steps_per_second': 2.143, 'total_flos': 2.682572850445824e+16, 'train_loss': 0.1198987268186083, 'epoch': 15.0})

In [18]:
trainer.save_model('ja_clf80')

### Eval

In [3]:
from transformers import AutoModelForTokenClassification
model_name = "ja_f1_best"

ner_model = AutoModelForTokenClassification.from_pretrained('ja_f1_best').eval()
tokenizer = AutoTokenizer.from_pretrained("daisaku-s/medtxt_ner_roberta")
re_model = AutoModelForSequenceClassification.from_pretrained('ja_clf80').eval()

idx2tag = ner_model.config.id2label
id2label = ['O', 'CAUSED', 'TREATMENT_FOR']

In [4]:
JA_DEV = Path('dev/ja_dev')
out_dir = Path('ft80_dev')
id2label = ['O', 'CAUSED', 'TREATMENT_FOR']
for f in JA_DEV.iterdir():
    if f.suffix != '.txt':
        continue
    with f.open() as text:
        annotations = []
        fields = text.readlines()[0].split(':')
        keyid = fields[0]
        offset = len(keyid)+1
        data = ':'.join(fields[1:])
        # ner
        with torch.inference_mode():
            vecs = tokenizer(data,
                             padding=True, 
                             truncation=True,
                             return_tensors="pt", max_length=512)
            ner_logits = ner_model(input_ids=vecs["input_ids"], attention_mask=vecs["attention_mask"])
            idx = torch.argmax(ner_logits.logits, dim=2).detach().cpu().numpy().tolist()[0]
            tokens = vecs.tokens()[1: -1]
        labels = [idx2tag[x] for x in idx][1:-1]
        prev_label = None
        prev_tag = ['', '']
        candidate = []
        start = 0
        for token, label in zip(tokens, labels):
            tag = label.split('-')
            if token.startswith('▁'):
                token = token[1:]
                start += 1
            if tag[0] == 'B':
                candidate= [(start, len(token) + start, token)]
            elif tag[0] != 'B' and len(prev_tag) > 1 and len(tag) > 1 and tag[1] == prev_tag[1]:
                candidate.append((start, len(token) + start, token))
            elif candidate:
                anno = ''.join(ctoken for s, e, ctoken in candidate)
                annotations.append((prev_tag[1], candidate[0][0] + offset - 1, candidate[-1][1] + offset - 1, anno))
                candidate = []
            start += len(token)
            prev_tag = tag
        count = 1
        new_annotations = []
        for ann in annotations:
            new_annotations.append(f"T{count}\t{ann[0]} {ann[1]} {ann[2]}\t{ann[3]}\n")
            count += 1
        # find all spans between the named entities
        classcorpus = []
        for a in new_annotations:
            for b in new_annotations:
                aid, ameta, _ = a.split('\t')
                bid, bmeta, _ = b.split('\t')
                if aid == bid:
                    continue
                alabel, astart, aend = ameta.split(' ')
                blabel, bstart, bend = bmeta.split(' ')
                if (alabel == 'DRUG' and blabel !='DRUG') or (alabel == 'DISORDER' and blabel != 'DRUG'):
                    start = min(int(aend), int(bend)) - offset
                    end = max(int(astart), int(bstart)) - offset
                    classcorpus.append([aid, bid, data[start: end]])
        # relation extraction
        rcount = 1
        for a,b, span in classcorpus:
            if span.strip():
                encoded_input = tokenizer(span, return_tensors='pt', max_length=512)
                with torch.inference_mode():
                    output = re_model(**encoded_input).logits
                    class_id = output.argmax().item()
                    if class_id > 0:
                        new_annotations.append(f"R{rcount}\t{id2label[class_id]} Arg1:{a} Arg2:{b}\n")
                        rcount += 1
        # save output
        with (out_dir / (f.stem + '.ann')).open('w') as output:
            for ann in new_annotations:
                output.write(ann)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


### Score


|type|tp|fp|fn|precision|recall|f1|fpm|fnm|
|---|---:|---:|---:|---:|---:|---:|---:|---:|
|CAUSED\|DISORDER\|DISORDER|1|29|28|0.0333|0.0345|0.0339|12|18|
|CAUSED\|DISORDER\|FUNCTION|0|3|5|0.0000|0.0000|0.0000|2|4|
|CAUSED\|DRUG\|DISORDER|43|67|146|0.3909|0.2275|0.2876|29|80|
|CAUSED\|DRUG\|FUNCTION|2|9|2|0.1818|0.5000|0.2667|3|1|
|TREATMENT_FOR\|DISORDER\|DISORDER|0|2|0|0.0000|0.0000|0.0000|2|0|
|TREATMENT_FOR\|DRUG\|DISORDER|1|9|37|0.1000|0.0263|0.0417|4|19|
|TREATMENT_FOR\|DRUG\|FUNCTION|0|0|1|0.0000|0.0000|0.0000|0|0|
|all|47|119|219|0.2831|0.1767|0.2176|52|122|


|||
|---|---:|
|Task2bMicroP| 0.2831|
|Task2bMicroR| 0.1767|
|Task2bMicroF1| 0.2176|
|Task2bMacroP|0.1009|
|Task2bMacroR| 0.1126|
|Task2bMacroF1| 0.0900|

## Train on train set and eval on dev set

model trained after the submission deadline

In [19]:
train_ds = Dataset.from_pandas(train_df, preserve_index=False)
train_ds = train_ds.map(preprocess_data, batched=True, remove_columns=train_ds.column_names)
train_ds.set_format("torch")

valid_ds = Dataset.from_pandas(dev_df, preserve_index=False)
valid_ds = valid_ds.map(preprocess_data, batched=True, remove_columns=valid_ds.column_names)
valid_ds.set_format("torch")


model = AutoModelForSequenceClassification.from_pretrained(model_checkpoint, num_labels=3, ignore_mismatched_sizes=True)

trainer = Trainer(
    model,
    args,
    train_dataset=train_ds,
    eval_dataset=valid_ds,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

trainer.train()

Map:   0%|          | 0/8497 [00:00<?, ? examples/s]

Map:   0%|          | 0/3225 [00:00<?, ? examples/s]

/home/ec2-user/anaconda3/envs/pytorch_p310/lib/python3.10/site-packages/huggingface_hub/file_download.py:1150: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at daisaku-s/medtxt_ner_roberta and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at daisaku-s/medtxt_ner_roberta and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([79, 768]) in the checkpoint and torch.Size([3, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([79]) in the ch

Epoch,Training Loss,Validation Loss,F1
1,0.266400,0.267047,0.433119
2,0.222600,0.280583,0.445134
3,0.191000,0.328285,0.474261
4,0.152700,0.358899,0.442407
5,0.133700,0.365120,0.442128
6,0.119100,0.394522,0.461780
7,0.099600,0.428971,0.451820
8,0.092500,0.412968,0.430608
9,0.084800,0.469531,0.429361
10,0.083900,0.462045,0.434892


TrainOutput(global_step=7980, training_loss=0.1157891875819156, metrics={'train_runtime': 3842.161, 'train_samples_per_second': 33.173, 'train_steps_per_second': 2.077, 'total_flos': 3.353512065652224e+16, 'train_loss': 0.1157891875819156, 'epoch': 15.0})

In [20]:
trainer.save_model('ja_clf_dev')

### Eval

In [5]:
from transformers import AutoModelForTokenClassification

ner_model = AutoModelForTokenClassification.from_pretrained('ja_best').eval()
tokenizer = AutoTokenizer.from_pretrained("daisaku-s/medtxt_ner_roberta")
re_model = AutoModelForSequenceClassification.from_pretrained('ja_clf_dev').eval()

idx2tag = ner_model.config.id2label
out_dir = Path('ft_dev')
JA_DEV = Path('dev/ja_dev')
id2label = ['O', 'CAUSED', 'TREATMENT_FOR']

In [6]:
for f in JA_DEV.iterdir():
    if f.suffix != '.txt':
        continue
    with f.open() as text:
        annotations = []
        fields = text.readlines()[0].split(':')
        keyid = fields[0]
        offset = len(keyid)+1
        data = ':'.join(fields[1:])
        # ner
        with torch.inference_mode():
            vecs = tokenizer(data,
                             padding=True, 
                             truncation=True,
                             return_tensors="pt", max_length=512)
            ner_logits = ner_model(input_ids=vecs["input_ids"], attention_mask=vecs["attention_mask"])
            idx = torch.argmax(ner_logits.logits, dim=2).detach().cpu().numpy().tolist()[0]
            tokens = vecs.tokens()[1: -1]
        labels = [idx2tag[x] for x in idx][1:-1]
        prev_label = None
        prev_tag = ['', '']
        candidate = []
        start = 0
        for token, label in zip(tokens, labels):
            tag = label.split('-')
            if token.startswith('▁'):
                token = token[1:]
                start += 1
            if tag[0] == 'B':
                candidate= [(start, len(token) + start, token)]
            elif tag[0] != 'B' and len(prev_tag) > 1 and len(tag) > 1 and tag[1] == prev_tag[1]:
                candidate.append((start, len(token) + start, token))
            elif candidate:
                anno = ''.join(ctoken for s, e, ctoken in candidate)
                annotations.append((prev_tag[1], candidate[0][0] + offset - 1, candidate[-1][1] + offset - 1, anno))
                candidate = []
            start += len(token)
            prev_tag = tag
        count = 1
        new_annotations = []
        for ann in annotations:
            new_annotations.append(f"T{count}\t{ann[0]} {ann[1]} {ann[2]}\t{ann[3]}\n")
            count += 1
        # find all spans between the named entities
        classcorpus = []
        for a in new_annotations:
            for b in new_annotations:
                aid, ameta, _ = a.split('\t')
                bid, bmeta, _ = b.split('\t')
                if aid == bid:
                    continue
                alabel, astart, aend = ameta.split(' ')
                blabel, bstart, bend = bmeta.split(' ')
                if (alabel == 'DRUG' and blabel !='DRUG') or (alabel == 'DISORDER' and blabel != 'DRUG'):
                    start = min(int(aend), int(bend)) - offset
                    end = max(int(astart), int(bstart)) - offset
                    classcorpus.append([aid, bid, data[start: end]])
        # relation extraction
        rcount = 1
        for a,b, span in classcorpus:
            if span.strip():
                encoded_input = tokenizer(span, return_tensors='pt', max_length=512)
                with torch.inference_mode():
                    output = re_model(**encoded_input).logits
                    class_id = output.argmax().item()
                    if class_id > 0:
                        new_annotations.append(f"R{rcount}\t{id2label[class_id]} Arg1:{a} Arg2:{b}\n")
                        rcount += 1
        # save output
        with (out_dir / (f.stem + '.ann')).open('w') as output:
            for ann in new_annotations:
                output.write(ann)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


### Score

|type|tp|fp|fn|precision|recall|f1|fpm|fnm|
|---|---:|---:|---:|---:|---:|---:|---:|---:|
|CAUSED\|DISORDER\|DISORDER|1|21|28|0.0455|0.0345|0.0392|4|16|
|CAUSED\|DISORDER\|FUNCTION|0|2|5|0.0000|0.0000|0.0000|1|3|
|CAUSED\|DRUG\|DISORDER|52|58|137|0.4727|0.2751|0.3478|22|76|
|CAUSED\|DRUG\|FUNCTION|3|9|1|0.2500|0.7500|0.3750|3|1|
|TREATMENT_FOR\|DRUG\|DISORDER|0|0|38|0.0000|0.0000|0.0000|0|20|
|TREATMENT_FOR\|DRUG\|FUNCTION|0|0|1|0.0000|0.0000|0.0000|0|0|
|all|56|90|210|0.3836|0.2105|0.2718|30|116|

|||
|---|---:|
|Task2bMicroP| 0.3836|
|Task2bMicroR| 0.2105|
|Task2bMicroF1| 0.2718|
|Task2bMacroP| 0.1280|
|Task2bMacroR| 0.1766|
|Task2bMacroF1| 0.1270|